# 01 - Data QC

**Presentation only.** Every transformation shown here lives in
`fluorescence_inference`; this notebook calls it and displays the result. If
you find yourself writing analysis in a cell, move it into the package
instead - the QC report has to be reproducible without Jupyter.

Prerequisite:

```bash
python scripts/export_processed_dataset.py --config configs/paired_100ms.yaml
```

Nothing here estimates a readout fidelity, an error rate or an atom-loss
probability. See `docs/DATA_AUDIT.md` for why this dataset cannot.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from fluorescence_inference.config import load_config
from fluorescence_inference.dataset import load_dataset
from fluorescence_inference import quality_control as qc

cfg = load_config("configs/paired_100ms.yaml", root=ROOT)
df, sites_df, meta = load_dataset(cfg)
summary = json.loads((cfg.reports_dir("qc", "qc_summary.json")).read_text())

pd.set_option("display.width", 140)
df.shape, sites_df.shape

## What is in the table

In [ ]:
display(pd.Series(summary["dataset"]).to_frame("value"))
display(pd.Series(summary["quality_flags"]).to_frame("rows"))
df.head(3)

## Site geometry

Sites are located from the shot-to-shot variance map, not the mean image.
The mean is dominated by a static fringe pattern; see `docs/DATA_AUDIT.md` §4.

In [ ]:
display(pd.DataFrame(summary["sites"]["geometry"]["grids"]).set_index("name").T)
flagged = sites_df[~sites_df["site_detected"].astype(bool)]
print(f"sites without a variance peak within the detection radius: {len(flagged)}")
flagged

## ROI placement, checked by eye

Both frames of the same shot, identical crop and identical intensity scale.

In [ ]:
for name in summary["figures"]:
    if name.startswith("qc_01"):
        display(Image(filename=str(cfg.reports_dir("qc", name))))

## The three measurement variants

| | definition |
|---|---|
| A | raw ROI sum |
| B | ROI sum minus the local annulus background |
| C | ROI sum minus a common mode measured on genuinely site-free pixels |

The dashed lines are descriptive reference levels for display, not a fitted
decision rule.

In [ ]:
display(Image(filename=str(cfg.reports_dir("qc", "qc_02_count_histograms.png"))))
display(Image(filename=str(cfg.reports_dir("qc", "qc_07_variant_comparison.png"))))

rows = []
for key, block in summary["variants"].items():
    for fid, v in block["per_frame"].items():
        rows.append({"variant": key, "frame": fid, "d_prime": v["separation_d_prime"],
                     "drift_per_100_shots": v["shot_order_slope_per_100_shots"],
                     "model_implied_overlap": v["model_implied_overlap"]})
pd.DataFrame(rows).round(3)

## Background behaviour

Frame 1 sits below frame 0 on every shot. The annulus shift is several times
the site-free shift, which is the signature of array-dependent light inside the
annulus rather than of a pure detector or stray-light offset.

In [ ]:
display(Image(filename=str(cfg.reports_dir("qc", "qc_04_background_drift.png"))))
display(Image(filename=str(cfg.reports_dir("qc", "qc_05_local_background_by_frame.png"))))
pd.json_normalize(summary["background"]).T

## Paired readout

Read the four regions as *apparent* regions. Frame-to-frame disagreement is a
statement about two measurements, not about an atom: this run has no
matched-empty, dark-frame or natural-loss control, so it cannot separate a lost
atom from a misclassified frame.

In [ ]:
display(Image(filename=str(cfg.reports_dir("qc", "qc_03_paired_scatter.png"))))
pr = summary["paired_readout"]
display(Markdown(
    f"- pairs: **{pr['n_pairs']:,}** from {summary['dataset']['n_shots']} shots\n"
    f"- agreement: **{pr['agreement']:.4f}** "
    f"(95% shot-cluster bootstrap {pr['agreement_ci95'][0]:.4f}-{pr['agreement_ci95'][1]:.4f})\n"
    f"- apparent bright-to-dark: **{pr['apparent_bright_to_dark_rate']:.4f}**\n"
    f"- apparent dark-to-bright: **{pr['apparent_dark_to_bright_rate']:.4f}**"))

## Site heterogeneity

In [ ]:
display(Image(filename=str(cfg.reports_dir("qc", "qc_06_site_maps.png"))))
site_stats = pd.read_csv(cfg.reports_dir("qc", "site_statistics.csv"))
f0 = site_stats[site_stats["frame_id"] == 0]
f0.groupby("grid")[["mean", "std", "local_background", "interdecile_spread"]].describe().T.round(1)

## Detector headroom and outliers

In [ ]:
display(Image(filename=str(cfg.reports_dir("qc", "qc_08_outliers.png"))))
display(pd.Series(summary["saturation"]).to_frame("value"))
display(Markdown("\n".join(f"- {n}" for n in summary["interpretation_guard"]["notes"])))